# 📦 Notebook 1 — Dataset Generation & Embedding Fundamentals

**What you'll learn**
- Generate realistic multi-domain datasets (clinical, reviews, failure logs, support tickets, research abstracts)
- Build TF-IDF and synthetic dense embeddings from scratch
- Understand *why* embedding quality matters before you measure similarity
- Visualise embedding space with **t-SNE** and **UMAP** — drill down by category

> **Run every cell top-to-bottom.** Outputs from this notebook are saved as `.npy` / `.csv` files and consumed by Notebooks 2 & 3.


## 1 · Imports & reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings, os, random, time
from pathlib import Path

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE

# umap (pip install umap-learn if missing)
try:
    import umap.umap_ as umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠  umap-learn not found – UMAP cells will be skipped")

# plotly for interactive drill-down
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

OUT = Path("embedding_outputs")
OUT.mkdir(exist_ok=True)
print("✅  Environment ready | Output dir:", OUT.resolve())


## 2 · Dataset Generation — 5 Real-World Domains

Each domain has **200–500 samples** across distinct sub-categories.  
We embed *all five* into a shared space later so you can see domain drift.


In [ ]:
# ─────────────────────────────────────────────────
# 2-A  CLINICAL DATASET  (500 records)
# ─────────────────────────────────────────────────
clinical_templates = {
    "cardiology": [
        "Patient presents with chest pain, elevated troponin, and ST-segment elevation consistent with STEMI.",
        "Echocardiogram shows reduced ejection fraction of 35% with anterolateral wall motion abnormality.",
        "Patient has history of hypertension, hyperlipidemia, and type 2 diabetes with recent angina.",
        "Coronary angiography revealed 90% stenosis of the LAD requiring urgent PCI.",
        "Post-MI patient on dual antiplatelet therapy showing good recovery with EF improving to 50%.",
        "Patient admitted with acute decompensated heart failure, bilateral crackles, and elevated BNP.",
        "Atrial fibrillation with rapid ventricular response, rate controlled with diltiazem infusion.",
        "Hypertensive emergency, BP 210/130, headache, blurred vision, renal function deteriorating.",
    ],
    "neurology": [
        "MRI brain shows acute ischemic infarct in the left MCA territory with cortical ribbon sign.",
        "Patient with sudden onset severe headache, worst of life, CT scan negative, LP confirms xanthochromia.",
        "Parkinson's disease with worsening tremor, bradykinesia, and postural instability despite levodopa.",
        "Epilepsy refractory to two AEDs, EEG shows focal onset from right temporal lobe.",
        "Multiple sclerosis relapse with optic neuritis, treated with IV methylprednisolone.",
        "Patient presents with ascending weakness and areflexia, EMG consistent with Guillain-Barré.",
        "Alzheimer's dementia with MMSE score 14/30, behavioural changes, and caregiver burden noted.",
        "Migraine with aura, frequency 10 per month, started on topiramate for prophylaxis.",
    ],
    "oncology": [
        "Stage IIIA non-small cell lung cancer, EGFR mutation positive, started on osimertinib.",
        "Breast cancer ER+ PR+ HER2-, node negative, proceeding with lumpectomy and radiation.",
        "Patient with colorectal cancer liver metastases, MSI-high, enrolled in immunotherapy trial.",
        "Pancreatic adenocarcinoma unresectable, CA 19-9 markedly elevated, palliative gemcitabine.",
        "Lymphoma with bulky mediastinal disease, FDG-PET shows widespread uptake, R-CHOP initiated.",
        "Post-bone-marrow transplant, day +45, tacrolimus adjusted for graft-versus-host prophylaxis.",
        "Prostate cancer Gleason 8, PSA 45, bone scan positive, enzalutamide started.",
        "Ovarian cancer recurrence, platinum-resistant, olaparib PARP inhibitor therapy considered.",
    ],
    "endocrinology": [
        "Type 1 diabetes in DKA, glucose 480, bicarbonate 12, anion gap 24, insulin infusion started.",
        "Hypothyroidism with TSH 45, levothyroxine dose uptitrated, patient symptomatic with fatigue.",
        "Cushing's syndrome confirmed by 24-hour UFC, pituitary MRI showing 6mm microadenoma.",
        "Primary hyperparathyroidism, calcium 11.8, PTH elevated, referred for parathyroidectomy.",
        "Polycystic ovary syndrome with insulin resistance, metformin and lifestyle modification advised.",
        "Adrenal insufficiency, morning cortisol 2.1, synacthen test flat, hydrocortisone replacement.",
        "Acromegaly with IGF-1 3x upper limit, pituitary macroadenoma, octreotide initiated.",
        "Type 2 diabetes HbA1c 10.2%, triple oral therapy failing, basal insulin added to regimen.",
    ],
    "respiratory": [
        "COPD exacerbation with FEV1 38% predicted, started on nebulised salbutamol and steroids.",
        "Community-acquired pneumonia, CRB-65 score 2, admitted for IV antibiotics and supportive care.",
        "Pulmonary embolism confirmed on CTPA, anticoagulation with LMWH, DOAC long-term.",
        "Severe asthma attack, peak flow 30% predicted, magnesium sulphate and ICU referral.",
        "Sarcoidosis stage II bilateral hilar lymphadenopathy and pulmonary infiltrates, steroids started.",
        "Interstitial lung disease on HRCT, UIP pattern, nintedanib commenced for IPF.",
        "Sleep apnoea with AHI 42, CPAP initiated, BMI 38, referred for weight management.",
        "Lung abscess right lower lobe, prolonged antibiotics, CT-guided drainage considered.",
    ]
}

def generate_clinical(n_per_cat=100):
    rows = []
    for cat, templates in clinical_templates.items():
        for i in range(n_per_cat):
            base = random.choice(templates)
            noise_words = random.sample(
                ["follow-up", "reviewed", "noted", "documented", "assessed",
                 "discussed with team", "plan updated", "labs pending"], 2)
            text = base + " " + " ".join(noise_words) + "."
            rows.append({"text": text, "category": cat, "domain": "clinical"})
    return pd.DataFrame(rows)

df_clinical = generate_clinical(100)
print(f"Clinical: {len(df_clinical)} rows | categories: {df_clinical.category.unique()}")
df_clinical.head(3)


In [ ]:
# ─────────────────────────────────────────────────
# 2-B  PRODUCT REVIEWS DATASET  (500 records)
# ─────────────────────────────────────────────────
review_templates = {
    "electronics": {
        "positive": [
            "This laptop is incredibly fast, the battery lasts all day, and the screen is stunning.",
            "Best smartphone I've ever owned — camera quality is exceptional in low light.",
            "The wireless earbuds have amazing noise cancellation and fit perfectly.",
            "Solid build quality, great performance for the price, highly recommend this tablet.",
        ],
        "negative": [
            "The laptop overheats after 30 minutes of use, very disappointing.",
            "Battery died after 3 months, terrible quality control on this phone.",
            "Earbuds kept disconnecting, Bluetooth range is laughably short.",
            "Screen cracked with minimal pressure, cheap materials despite premium price.",
        ],
        "neutral": [
            "Decent laptop, does the job, nothing spectacular but no major issues.",
            "Average phone, camera is okay, battery life could be better.",
        ]
    },
    "restaurants": {
        "positive": [
            "Absolutely incredible meal, the pasta was perfectly cooked and service was attentive.",
            "Best sushi in the city, fish is so fresh, ambiance is beautiful.",
            "Lovely brunch spot, avocado toast was divine, coffee was excellent.",
            "Outstanding steakhouse, perfectly cooked ribeye, wine list is superb.",
        ],
        "negative": [
            "Waited 45 minutes for cold food, the steak was overcooked and staff was rude.",
            "Overpriced and underwhelming, pasta was mushy and portion sizes tiny.",
            "Terrible service, had to ask three times for water, will not return.",
            "Food poisoning after dining here, health standards seem non-existent.",
        ],
        "neutral": [
            "Decent neighbourhood spot, nothing special but food was acceptable.",
            "Average pizza, service was fine, will probably not go out of my way to return.",
        ]
    },
    "hotels": {
        "positive": [
            "Stunning hotel with breathtaking views, impeccable service and luxurious rooms.",
            "Perfect stay, staff went above and beyond, breakfast was exceptional.",
            "Beautiful boutique hotel, incredibly comfortable beds, great location.",
        ],
        "negative": [
            "Room was dirty, plumbing was broken, staff completely unhelpful.",
            "Noisy construction next door all night, refund request denied, appalling.",
            "Check-in took over an hour, room not ready despite late arrival guarantee.",
        ],
        "neutral": [
            "Standard business hotel, clean but lacks character, convenient location.",
            "Average stay, nothing to complain about but nothing special either.",
        ]
    },
    "software": {
        "positive": [
            "This app has completely transformed my workflow, intuitive and powerful.",
            "Excellent project management tool, the collaboration features are outstanding.",
            "Best note-taking app available, syncs perfectly across all my devices.",
        ],
        "negative": [
            "App crashes constantly, lost hours of work, terrible stability.",
            "Subscription price doubled overnight with no notice, customer support non-existent.",
            "Buggy interface, features don't work as advertised, waste of money.",
        ],
        "neutral": [
            "Gets the job done, some quirks but generally works as expected.",
            "Average tool, does what it says, nothing revolutionary.",
        ]
    }
}

def generate_reviews(n_per_cat=125):
    rows = []
    for cat, sentiments in review_templates.items():
        for sentiment, templates in sentiments.items():
            count = n_per_cat // len(sentiments)
            for i in range(count):
                text = random.choice(templates)
                extras = random.choice(["Would recommend.", "Worth every penny.", 
                                        "Returning customer.", "Will not return.",
                                        "Mixed feelings overall."])
                rows.append({"text": text + " " + extras,
                             "category": cat, "sentiment": sentiment, "domain": "reviews"})
    return pd.DataFrame(rows)

df_reviews = generate_reviews(125)
print(f"Reviews: {len(df_reviews)} rows | categories: {df_reviews.category.unique()}")
df_reviews.head(3)


In [ ]:
# ─────────────────────────────────────────────────
# 2-C  EQUIPMENT FAILURE REASONS  (400 records)
# ─────────────────────────────────────────────────
failure_templates = {
    "mechanical": [
        "Bearing failure due to inadequate lubrication causing excessive heat and vibration.",
        "Gear tooth fracture from metal fatigue after 15,000 operating hours beyond maintenance interval.",
        "Shaft misalignment detected, causing seal wear and subsequent hydraulic fluid leakage.",
        "Impeller blade corrosion in pump assembly, flow rate reduced by 60%, cavitation observed.",
        "Bolt shear failure in mounting bracket due to repeated cyclic loading stress concentration.",
        "Valve stem breakage in high-pressure system, emergency shutdown triggered automatically.",
    ],
    "electrical": [
        "Motor winding insulation breakdown caused by thermal overload during peak demand period.",
        "Control board failure due to moisture ingress in outdoor enclosure, corrosion on PCB traces.",
        "Power supply capacitor failure causing voltage ripple and downstream equipment instability.",
        "Ground fault in cable tray due to insulation degradation from UV exposure over time.",
        "Contactor welding from repeated high-inrush switching, circuit protection did not trip.",
        "Sensor calibration drift after 3 years, false readings triggered unnecessary shutdowns.",
    ],
    "software": [
        "SCADA system deadlock caused by concurrent write operations to shared memory buffer.",
        "PLC ladder logic race condition, output coil toggling at high frequency damaging actuator.",
        "Firmware update corrupted bootloader, device unable to restart without factory reset.",
        "Database query timeout under load causing HMI freeze and loss of operator visibility.",
        "Memory leak in historian software, system RAM exhausted after 72-hour continuous run.",
        "Cybersecurity intrusion through unpatched OPC-UA server, ransomware encrypted historian.",
    ],
    "thermal": [
        "Heat exchanger fouling reduced thermal efficiency by 40%, process temperature out of spec.",
        "Cooling tower fan motor failure, process chiller temperature exceeded safe operating limit.",
        "Refractory lining erosion in kiln, shell temperature alarmingly elevated, unplanned outage.",
        "Freeze protection failure during winter, water in instrumentation lines causing cracked fittings.",
        "Exothermic reaction runaway in reactor, pressure relief valve activated, batch scrapped.",
        "Thermocouple failure in furnace, no temperature feedback, product quality non-conformance.",
    ],
    "structural": [
        "Stress corrosion cracking found in pressure vessel during scheduled inspection, retirement.",
        "Scaffold collapse due to overloading, incident report filed, OSHA notification required.",
        "Pipeline wall thinning from internal corrosion, ultrasonic testing revealed critical defect.",
        "Foundation settlement causing misalignment of rotating equipment, vibration levels exceeded.",
        "Weld defect in structural support beam identified during NDT, repair weld completed.",
        "Tank floor corrosion from bottom-water accumulation, leak detected during integrity test.",
    ]
}

def generate_failures(n_per_cat=80):
    rows = []
    severity = ["critical", "major", "minor"]
    for cat, templates in failure_templates.items():
        for i in range(n_per_cat):
            text = random.choice(templates)
            sev = random.choice(severity)
            downtime = random.randint(1, 168)
            rows.append({"text": f"[{sev.upper()}] {text} Estimated downtime: {downtime}h.",
                         "category": cat, "severity": sev, "downtime_hours": downtime,
                         "domain": "failures"})
    return pd.DataFrame(rows)

df_failures = generate_failures(80)
print(f"Failures: {len(df_failures)} rows | categories: {df_failures.category.unique()}")
df_failures.head(3)


In [ ]:
# ─────────────────────────────────────────────────
# 2-D  SUPPORT TICKETS  (400 records)
# ─────────────────────────────────────────────────
ticket_templates = {
    "authentication": [
        "Unable to log in after password reset, reset email not received despite multiple attempts.",
        "Two-factor authentication not working, TOTP codes rejected, account locked out.",
        "Single sign-on integration broken after AD password change, all users affected.",
        "OAuth token expiry too aggressive, users logged out every 15 minutes, productivity hit.",
        "Forgot password flow broken in production, reset link returns 404 error.",
    ],
    "billing": [
        "Charged twice for the same subscription this month, need immediate refund.",
        "Invoice shows incorrect VAT rate, company requires corrected invoice for accounting.",
        "Subscription downgrade not reflected in billing, still charged at Pro rate.",
        "Payment method update not saving, getting error 'card declined' on valid card.",
        "Trial extended but credit card was still charged, confused and frustrated.",
    ],
    "performance": [
        "Dashboard takes over 60 seconds to load, was instant before recent deployment.",
        "Report generation timing out for datasets over 10,000 rows, was working last week.",
        "API response times degraded from 200ms to 8 seconds under normal load.",
        "Mobile app extremely slow on 4G, suspected memory leak causing progressive slowdown.",
        "Search functionality returns results in 30 seconds, completely unusable for our team.",
    ],
    "data": [
        "Exported CSV has corrupted UTF-8 characters, special characters appear as question marks.",
        "Data import failed silently, no error shown but records not appearing in system.",
        "Historical data missing after database migration last weekend, 3 months of records gone.",
        "Duplicate records appearing after bulk import, deduplication tool not working correctly.",
        "Real-time sync between our CRM and your platform stopped working 2 days ago.",
    ],
    "ui_ux": [
        "Submit button disabled on checkout form despite all required fields being completed.",
        "Dark mode setting not persisting between sessions, resets to light mode on refresh.",
        "Accessibility issue: screen reader not announcing modal dialogs, affects visually impaired users.",
        "Date picker broken on Safari iOS 17, cannot select dates in the booking flow.",
        "Pagination not working on the reports page, cannot navigate past page 1 of results.",
    ]
}

def generate_tickets(n_per_cat=80):
    rows = []
    priorities = ["P1-Critical", "P2-High", "P3-Medium", "P4-Low"]
    weights = [0.1, 0.3, 0.4, 0.2]
    for cat, templates in ticket_templates.items():
        for i in range(n_per_cat):
            text = random.choice(templates)
            priority = random.choices(priorities, weights)[0]
            rows.append({"text": f"[{priority}] {text}",
                         "category": cat, "priority": priority, "domain": "support"})
    return pd.DataFrame(rows)

df_tickets = generate_tickets(80)
print(f"Tickets: {len(df_tickets)} rows | categories: {df_tickets.category.unique()}")
df_tickets.head(3)


In [ ]:
# ─────────────────────────────────────────────────
# 2-E  RESEARCH ABSTRACTS  (300 records)
# ─────────────────────────────────────────────────
abstract_templates = {
    "machine_learning": [
        "We propose a novel attention mechanism that achieves state-of-the-art performance on NLP benchmarks.",
        "Federated learning framework enabling privacy-preserving model training across distributed nodes.",
        "Graph neural network approach for molecular property prediction surpasses existing baselines.",
        "Contrastive self-supervised learning significantly reduces labelled data requirements.",
        "Mixture-of-experts architecture scales to trillion parameters with sub-linear inference cost.",
    ],
    "climate_science": [
        "Analysis of Arctic sea ice extent shows accelerated decline correlated with global temperature.",
        "Ocean acidification rate measurements confirm pH drop of 0.1 units since industrial revolution.",
        "Carbon capture efficiency of novel MOF sorbent exceeds 95% at 400ppm CO2 concentration.",
        "Permafrost thaw releasing methane at rates 50% higher than IPCC 2021 projections.",
        "Renewable energy transition modelling shows 1.5°C target achievable with current technology.",
    ],
    "genomics": [
        "Whole-genome sequencing of 50,000 individuals identifies 127 novel disease-associated loci.",
        "CRISPR base editing corrects pathogenic variant in haemophilia A murine model.",
        "Single-cell RNA sequencing reveals previously unknown immune cell subpopulation in tumours.",
        "Polygenic risk score for type 2 diabetes achieves AUC 0.84 in prospective validation cohort.",
        "Long-read sequencing resolves complex structural variants in centromeric regions.",
    ],
    "materials_science": [
        "Perovskite solar cell efficiency reaches 29.8% with novel passivation layer technique.",
        "High-entropy alloy demonstrates exceptional fatigue resistance at cryogenic temperatures.",
        "Graphene aerogel achieves world record thermal insulation with density of 0.16 mg/cm³.",
        "Solid-state electrolyte enables lithium-metal batteries with 1000 charge cycle stability.",
        "Self-healing polymer restores 95% tensile strength after complete fracture in 24 hours.",
    ],
    "epidemiology": [
        "Cohort study of 200,000 participants links ultra-processed food consumption to dementia risk.",
        "Vaccine effectiveness against severe disease remains above 80% at 12 months post-booster.",
        "Air pollution exposure in early childhood associated with 15% increased asthma prevalence.",
        "Antimicrobial resistance modelling predicts 10 million annual deaths by 2050 without intervention.",
        "Social determinants of health account for 35% of variation in cardiovascular mortality.",
    ]
}

def generate_abstracts(n_per_cat=60):
    rows = []
    for cat, templates in abstract_templates.items():
        for i in range(n_per_cat):
            text = random.choice(templates)
            methods = random.choice(["Study design: RCT.", "Methods: retrospective cohort.", 
                                     "Approach: meta-analysis.", "Design: prospective observational.",
                                     "Methods: computational modelling."])
            rows.append({"text": f"{text} {methods}",
                         "category": cat, "domain": "research"})
    return pd.DataFrame(rows)

df_abstracts = generate_abstracts(60)
print(f"Abstracts: {len(df_abstracts)} rows | categories: {df_abstracts.category.unique()}")
df_abstracts.head(3)


In [ ]:
# ── Combine all domains ───────────────────────────────────────────────
df_all = pd.concat([
    df_clinical[["text","category","domain"]],
    df_reviews[["text","category","domain"]],
    df_failures[["text","category","domain"]],
    df_tickets[["text","category","domain"]],
    df_abstracts[["text","category","domain"]],
], ignore_index=True)

df_all["domain_category"] = df_all["domain"] + " › " + df_all["category"]
print(f"\n📊 COMBINED DATASET: {len(df_all)} records across {df_all.domain.nunique()} domains "
      f"and {df_all.category.nunique()} categories")
print("\nSamples per domain:")
print(df_all.groupby("domain").size().to_string())

# Save raw data
df_all.to_csv(OUT / "all_datasets.csv", index=False)
df_clinical.to_csv(OUT / "clinical.csv", index=False)
df_failures.to_csv(OUT / "failures.csv", index=False)
print("\n✅  Saved to", OUT)


## 3 · Building Embeddings

We create **two types** of embeddings:
1. **TF-IDF + SVD** (Latent Semantic Analysis) — classical, no ML required, great baseline
2. **Synthetic dense embeddings** — simulates transformer output; replace with `sentence-transformers` when you have GPU/internet

Each approach produces a dense matrix `(N × D)` ready for similarity computation.


In [ ]:
# ─────────────────────────────────────────────────
# 3-A  TF-IDF → SVD (LSA)  — 256 dimensions
# ─────────────────────────────────────────────────
print("Building TF-IDF vectoriser …")
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)
X_tfidf = tfidf.fit_transform(df_all["text"])
print(f"  TF-IDF sparse matrix: {X_tfidf.shape}")

print("Reducing to 256 dims with TruncatedSVD (LSA) …")
svd = TruncatedSVD(n_components=256, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)
X_lsa_norm = normalize(X_lsa, norm="l2")   # unit-length vectors
print(f"  LSA embeddings: {X_lsa_norm.shape} | "
      f"variance explained: {svd.explained_variance_ratio_.sum():.2%}")

# Domain-specific embedders (trained only on that domain)
print("\nBuilding per-domain TF-IDF embeddings …")
domain_embeddings = {}
for domain in df_all.domain.unique():
    mask = df_all.domain == domain
    vect = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
    X_d = vect.fit_transform(df_all.loc[mask, "text"])
    svd_d = TruncatedSVD(n_components=min(64, X_d.shape[1]-1), random_state=42)
    X_e = normalize(svd_d.fit_transform(X_d), norm="l2")
    domain_embeddings[domain] = (mask, X_e)
    print(f"  {domain:12s} → {X_e.shape}")

np.save(OUT / "embeddings_lsa.npy", X_lsa_norm)
print("\n✅  LSA embeddings saved")


In [ ]:
# ─────────────────────────────────────────────────
# 3-B  Simulate dense transformer embeddings
#      Replace this block with:
#        from sentence_transformers import SentenceTransformer
#        model = SentenceTransformer('all-MiniLM-L6-v2')
#        X_dense = model.encode(df_all.text.tolist(), batch_size=64)
# ─────────────────────────────────────────────────
print("Simulating dense embeddings (384-dim like all-MiniLM-L6-v2) …")
print("  ► Replace with real sentence-transformers for production use!")

le_domain = LabelEncoder().fit(df_all.domain)
le_cat = LabelEncoder().fit(df_all.category)

domain_codes = le_domain.transform(df_all.domain)
cat_codes = le_cat.transform(df_all.category)

n = len(df_all)
D = 384

# Cluster centres per (domain, category) pair
rng = np.random.RandomState(0)
n_clusters = df_all.domain_category.nunique()
cluster_centres = rng.randn(n_clusters, D) * 3

le_dc = LabelEncoder().fit(df_all.domain_category)
dc_codes = le_dc.transform(df_all.domain_category)

X_dense_raw = cluster_centres[dc_codes] + rng.randn(n, D) * 0.4
X_dense = normalize(X_dense_raw, norm="l2")

np.save(OUT / "embeddings_dense.npy", X_dense)
print(f"  Dense embeddings: {X_dense.shape}")
print("✅  Both embedding matrices ready")


## 4 · Visualise Embedding Space — t-SNE

t-SNE collapses high-dimensional embeddings to 2-D for inspection.  
**Use it to verify** that same-category texts cluster together.


In [ ]:
# ─────────────────────────────────────────────────
# 4-A  t-SNE on LSA embeddings
# ─────────────────────────────────────────────────
print("Running t-SNE on LSA embeddings (this takes ~30s) …")
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000,
            learning_rate="auto", init="pca", random_state=42)
X_tsne_lsa = tsne.fit_transform(X_lsa_norm)
print(f"  Done in {time.time()-t0:.1f}s")

df_plot = df_all.copy()
df_plot["tsne_x"] = X_tsne_lsa[:, 0]
df_plot["tsne_y"] = X_tsne_lsa[:, 1]
df_plot.to_csv(OUT / "tsne_lsa_coords.csv", index=False)
print("  Saved coords →", OUT / "tsne_lsa_coords.csv")


In [ ]:
# ─────────────────────────────────────────────────
# 4-B  t-SNE STATIC PLOT — coloured by domain
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

domain_colors = {
    "clinical": "#E63946", "reviews": "#457B9D",
    "failures": "#F4A261", "support": "#2A9D8F", "research": "#8338EC"
}

for ax, col_by, title in [
    (axes[0], "domain", "t-SNE (LSA) — coloured by DOMAIN"),
    (axes[1], "category", "t-SNE (LSA) — coloured by CATEGORY"),
]:
    unique_vals = df_plot[col_by].unique()
    cmap = cm.get_cmap("tab20", len(unique_vals))
    for idx, val in enumerate(sorted(unique_vals)):
        mask = df_plot[col_by] == val
        color = domain_colors.get(val, cmap(idx))
        ax.scatter(df_plot.loc[mask, "tsne_x"], df_plot.loc[mask, "tsne_y"],
                   c=[color], label=val, s=12, alpha=0.65, linewidths=0)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, markerscale=2)
    ax.grid(alpha=0.3)

plt.suptitle("Embedding Quality Check — LSA Embeddings", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUT / "tsne_lsa_static.png", dpi=150, bbox_inches="tight")
plt.show()
print("💡 TIP: Clear domain-level separation confirms embeddings encode semantic meaning.")


In [ ]:
# ─────────────────────────────────────────────────
# 4-C  INTERACTIVE t-SNE — drill-down by domain/category
# ─────────────────────────────────────────────────
fig = px.scatter(
    df_plot, x="tsne_x", y="tsne_y",
    color="domain", symbol="domain",
    hover_data={"category": True, "text": True,
                "tsne_x": False, "tsne_y": False},
    title="Interactive t-SNE — LSA Embeddings (hover to read text)",
    width=1000, height=650,
    color_discrete_map=domain_colors,
    opacity=0.7
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(
    legend=dict(title="Domain", itemsizing="constant"),
    font=dict(size=11)
)
fig.write_html(OUT / "tsne_interactive.html")
fig.show()
print("✅  Interactive plot saved → open tsne_interactive.html for full drill-down")


In [ ]:
# ─────────────────────────────────────────────────
# 4-D  DRILL-DOWN — single domain at a time
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

domains_to_plot = list(df_all.domain.unique())
cmap_list = ["Reds", "Blues", "Oranges", "Greens", "Purples"]

for i, (domain, cmap_name) in enumerate(zip(domains_to_plot, cmap_list)):
    ax = axes[i]
    mask = df_plot.domain == domain
    sub = df_plot[mask]
    cats = sub.category.unique()
    cmap = cm.get_cmap(cmap_name, len(cats))
    for j, cat in enumerate(sorted(cats)):
        cm_mask = sub.category == cat
        ax.scatter(sub.loc[cm_mask, "tsne_x"], sub.loc[cm_mask, "tsne_y"],
                   c=[cmap(j+1)], label=cat, s=18, alpha=0.8, linewidths=0)
    ax.set_title(f"Domain: {domain.upper()}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=7, loc="best", markerscale=2)
    ax.grid(alpha=0.3)
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")

axes[-1].set_visible(False)
plt.suptitle("t-SNE Drill-Down by Category within Each Domain", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "tsne_drilldown_domains.png", dpi=150, bbox_inches="tight")
plt.show()


## 5 · UMAP — Faster & Preserves Global Structure

UMAP is generally preferred over t-SNE for large datasets:  
- **Faster** at scale  
- **Preserves global cluster structure** (not just local neighbours)
- Better for ANN index validation


In [ ]:
if HAS_UMAP:
    print("Running UMAP on LSA embeddings …")
    t0 = time.time()
    reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1,
                        metric="cosine", random_state=42)
    X_umap = reducer.fit_transform(X_lsa_norm)
    print(f"  Done in {time.time()-t0:.1f}s")

    df_plot["umap_x"] = X_umap[:, 0]
    df_plot["umap_y"] = X_umap[:, 1]
    df_plot.to_csv(OUT / "tsne_lsa_coords.csv", index=False)

    fig = px.scatter(
        df_plot, x="umap_x", y="umap_y",
        color="domain", hover_data={"category": True, "text": True,
                                    "umap_x": False, "umap_y": False},
        title="Interactive UMAP — LSA Embeddings",
        width=1000, height=650, color_discrete_map=domain_colors, opacity=0.7
    )
    fig.update_traces(marker=dict(size=5))
    fig.write_html(OUT / "umap_interactive.html")
    fig.show()
    print("✅  UMAP plot saved → umap_interactive.html")
else:
    print("⚠  UMAP not installed. Run:  pip install umap-learn")
    print("   Skipping UMAP visualisation.")


## 6 · Embedding Health Diagnostics

Before running ANN search, verify your embeddings are well-formed.


In [ ]:
# ─────────────────────────────────────────────────
# 6-A  Intra-class vs Inter-class distance
# ─────────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

print("Computing intra vs inter-class cosine similarity per domain …\n")
results = []

for domain in df_all.domain.unique():
    mask = (df_all.domain == domain).values
    X_d = X_lsa_norm[mask]
    
    # intra: mean similarity between same-category pairs (sample 200)
    cats = df_all.loc[mask, "category"].values
    intra_sims, inter_sims = [], []
    
    for cat in np.unique(cats):
        idx = np.where(cats == cat)[0]
        if len(idx) > 1:
            s = min(len(idx), 30)
            chosen = np.random.choice(idx, s, replace=False)
            sim_mat = cosine_similarity(X_d[chosen])
            np.fill_diagonal(sim_mat, np.nan)
            intra_sims.append(np.nanmean(sim_mat))
    
    # inter: pairs from different categories
    idx_a = np.random.choice(len(X_d), 100, replace=False)
    idx_b = np.random.choice(len(X_d), 100, replace=False)
    diff_mask = cats[idx_a] != cats[idx_b]
    if diff_mask.sum() > 10:
        sims = (X_d[idx_a[diff_mask]] * X_d[idx_b[diff_mask]]).sum(axis=1)
        inter_sims = sims.tolist()
    
    results.append({
        "domain": domain,
        "intra_class_sim": np.mean(intra_sims) if intra_sims else 0,
        "inter_class_sim": np.mean(inter_sims) if inter_sims else 0,
    })

df_health = pd.DataFrame(results)
df_health["separation_ratio"] = df_health["intra_class_sim"] / (df_health["inter_class_sim"] + 1e-9)

print(df_health.to_string(index=False))
print("\n💡 Higher separation_ratio = better embedding quality for that domain")
print("   Ratio > 1.5 is generally good for retrieval tasks")


In [ ]:
# ─────────────────────────────────────────────────
# 6-B  Visualise health as bar chart
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = df_health.domain
width = 0.35
x_pos = np.arange(len(x))

axes[0].bar(x_pos - width/2, df_health.intra_class_sim, width, label="Intra-class", color="#2A9D8F")
axes[0].bar(x_pos + width/2, df_health.inter_class_sim, width, label="Inter-class", color="#E76F51")
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(x, rotation=20)
axes[0].set_ylabel("Mean Cosine Similarity")
axes[0].set_title("Intra vs Inter-class Similarity")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.4)

axes[1].bar(x_pos, df_health.separation_ratio, color="#457B9D", edgecolor="white")
axes[1].axhline(1.5, color="red", linestyle="--", label="Good threshold (1.5)")
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(x, rotation=20)
axes[1].set_ylabel("Separation Ratio")
axes[1].set_title("Embedding Separation Ratio by Domain")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.4)

plt.suptitle("Embedding Health Diagnostics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "embedding_health.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# 6-C  Confusion matrix — top-1 category retrieval accuracy
# ─────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

domain = "clinical"
mask = (df_all.domain == domain).values
X_d = X_lsa_norm[mask]
y_true = df_all.loc[mask, "category"].values
cats = np.unique(y_true)

# For each sample, find its nearest neighbour (excluding self)
sim_matrix = cosine_similarity(X_d)
np.fill_diagonal(sim_matrix, -1)
nn_indices = sim_matrix.argmax(axis=1)
y_pred = y_true[nn_indices]

cm = confusion_matrix(y_true, y_pred, labels=cats)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=cats, yticklabels=cats,
            cmap="Blues", ax=ax)
ax.set_title(f"Nearest-Neighbour Retrieval Confusion Matrix
(Domain: {domain})", fontsize=12)
ax.set_xlabel("Predicted Category"); ax.set_ylabel("True Category")
plt.tight_layout()
plt.savefig(OUT / "nn_confusion_matrix.png", dpi=150)
plt.show()

accuracy = (y_true == y_pred).mean()
print(f"\nTop-1 NN accuracy ({domain}): {accuracy:.1%}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=cats))


In [ ]:
print("=" * 60)
print("NOTEBOOK 1 COMPLETE — Generated files:")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name}")
print("\n→ Proceed to Notebook 2: Similarity Methods Comparison")
print("→ Proceed to Notebook 3: ANN Search & Retrieval")
